In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

# ---------------------------------
# LOAD DATA
# ---------------------------------

df = pd.read_csv("../data/train_transaction.csv")

# sample for speed
df = df.sample(100000, random_state=42)

# ---------------------------------
# CREATE TIME-BASED SPLIT
# ---------------------------------

# Earlier data
train_df = df.iloc[:70000]

# Later data
test_df = df.iloc[70000:]

target = "isFraud"

# ---------------------------------
# SIMULATE DRIFT
# ---------------------------------

# Increase fraud probability in later data
fraud_rows = test_df[test_df[target] == 1]

# shift feature distributions
if "TransactionAmt" in test_df.columns:
    test_df.loc[fraud_rows.index, "TransactionAmt"] *= 2

if "card1" in test_df.columns:
    test_df.loc[fraud_rows.index, "card1"] += 5000

print("Drift simulation completed")

Drift simulation completed


In [2]:
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

# numeric only for simplicity
X_train = X_train.select_dtypes(include=np.number).fillna(0)
X_test = X_test.select_dtypes(include=np.number).fillna(0)

model = XGBClassifier(
    eval_metric='logloss'
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

auc_before = roc_auc_score(y_test, y_prob)

print("AUC before retraining:", auc_before)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99     28919
           1       0.83      0.37      0.51      1081

    accuracy                           0.97     30000
   macro avg       0.90      0.68      0.75     30000
weighted avg       0.97      0.97      0.97     30000

AUC before retraining: 0.8998847589837435


In [3]:
# ---------------------------------
# RETRAINING STRATEGY
# ---------------------------------

RECALL_THRESHOLD = 0.85
CURRENT_RECALL = 0.78

days_since_last_training = 35
PERIODIC_LIMIT = 30

retrain = False

# threshold-based trigger
if CURRENT_RECALL < RECALL_THRESHOLD:
    print("Recall dropped below threshold")
    retrain = True

# periodic trigger
if days_since_last_training > PERIODIC_LIMIT:
    print("Periodic retraining triggered")
    retrain = True

if retrain:
    print("Retraining model...")
    
    retrained_model = XGBClassifier(
        eval_metric='logloss'
    )

    retrained_model.fit(X_test, y_test)

    retrained_pred = retrained_model.predict(X_test)
    retrained_prob = retrained_model.predict_proba(X_test)[:, 1]

    auc_after = roc_auc_score(y_test, retrained_prob)

    print("AUC after retraining:", auc_after)

else:
    print("No retraining needed")

Recall dropped below threshold
Periodic retraining triggered
Retraining model...
AUC after retraining: 0.9997995293818689
